In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    apply_custom_style,
    collect_attributions,
    load_dyst_samples,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
@dataclass
class HeadSharpnessConfig:
    device: str
    work_dir: str
    data_dir: str
    plot_save_dir: str
    split_name: str
    subsample_interval: int
    system_name: str
    context_length: int
    context_start_time: int
    prediction_length: int
    sample_idx: int
    selected_dim: int


def make_head_sharpness_config(
    device: str | None = None,
    work_dir: str | None = None,
    split_name: str = "base40",
    subsample_interval: int = 1,
    system_name: str = "Lorenz",
    context_length: int = 512,
    context_start_time: int = 2000,
    prediction_length: int = 512,
    sample_idx: int = 0,
    selected_dim: int = 0,
) -> HeadSharpnessConfig:
    dev = device if device is not None else ("cuda:0" if torch.cuda.is_available() else "cpu")
    wd = work_dir if work_dir is not None else os.getenv("WORK", "/work")
    dd = os.path.join(wd, "data")
    psd = os.path.join("../../figs", "head_outputs")
    os.makedirs(psd, exist_ok=True)
    return HeadSharpnessConfig(
        device=dev,
        work_dir=wd,
        data_dir=dd,
        plot_save_dir=psd,
        split_name=split_name,
        subsample_interval=subsample_interval,
        system_name=system_name,
        context_length=context_length,
        context_start_time=context_start_time,
        prediction_length=prediction_length,
        sample_idx=sample_idx,
        selected_dim=selected_dim,
    )


cfg = make_head_sharpness_config()

In [ ]:
pipeline = CircuitLensChronos.from_pretrained(
    "amazon/chronos-t5-base", device_map=cfg.device, torch_dtype=torch.bfloat16
)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
split_dir = os.path.join(cfg.data_dir, cfg.split_name)
context_end_time = cfg.context_start_time + cfg.context_length

dyst_coords = load_dyst_samples(cfg.system_name, split_dir, one_dim_target=False, num_samples=1)
dyst_coords = torch.tensor(dyst_coords[cfg.sample_idx, cfg.selected_dim, :]).unsqueeze(0)
dyst_coords = dyst_coords[:, :: cfg.subsample_interval]
context = dyst_coords[:, cfg.context_start_time : context_end_time]

print(context.shape)

future_vals = dyst_coords[:, context_end_time : context_end_time + cfg.prediction_length]
print(f"future_vals shape: {future_vals.shape}")


In [ ]:
pipeline.add_v_attribution_hooks([(i, j) for j in range(num_heads) for i in range(num_layers)])
pipeline.add_head_attribution_hooks([(i, j) for j in range(num_heads) for i in range(num_layers)])

preds = pipeline.predict_with_full_outputs(
    context, prediction_length=cfg.prediction_length, output_attentions=True, return_dict_in_generate=True
)

In [ ]:
def collect_ca_attention_scores(preds):
    dec_outs = preds[1]
    rollouts = len(dec_outs)

    num_layers = len(dec_outs[0].cross_attentions[0])
    num_samples, num_heads, _, context_len = dec_outs[0].cross_attentions[0][0].shape

    t = (rollouts - 1) * 64 + len(dec_outs[-1].cross_attentions)
    attention_scores = torch.zeros((t, num_layers, num_samples, num_heads, context_len))

    t_curr = 0
    for r in range(rollouts):
        for step in dec_outs[r].cross_attentions:
            for l in range(num_layers):
                attention_scores[t_curr, l] = step[l][:, :, -1, :]
            t_curr += 1

    return attention_scores


def collect_sa_attention_scores(preds):
    dec_outs = preds[1]
    rollouts = len(dec_outs)

    num_layers = len(dec_outs[0].cross_attentions[0])
    num_samples, num_heads, _, max_context_len = dec_outs[0].cross_attentions[-11][0].shape

    t = (rollouts - 1) * 64 + len(dec_outs[-1].cross_attentions)
    attention_scores = torch.empty((t, num_layers, num_samples, num_heads, max_context_len))

    t_curr = 0
    for r in range(rollouts):
        for step in dec_outs[r].cross_attentions:
            for l in range(num_layers):
                context_len_curr = step[l][:, :, -1, :].shape[-1]
                attention_scores[t_curr, l, :, :, :context_len_curr] = step[l][:, :, -1, :]
            t_curr += 1

    return attention_scores

In [ ]:
sample0 = collect_ca_attention_scores(preds)[:, :, 0, :, :]
sample0_sa = collect_sa_attention_scores(preds)[:, :, 0, :, :]

In [ ]:
def plot_context_series(context: torch.Tensor) -> None:
    context_np = context.squeeze(0).detach().cpu().numpy()
    xs = np.arange(context_np.shape[-1])
    plt.figure(figsize=(10, 3))
    plt.plot(xs, context_np, linewidth=2, color="black")
    plt.xlabel("Timestep", fontweight="bold")
    plt.title("Context", fontweight="bold")
    plt.tight_layout()


plot_context_series(context)

In [ ]:
import math

from matplotlib import gridspec


def plot_attention_heatmap_all_heads(
    sample0: torch.Tensor,
    plot_save_dir: str,
    *,
    layer: int,
    num_layers: int,
    num_heads: int,
) -> None:
    assert sample0.ndim == 4, f"Expected 4D tensor [t, layers, heads, context], got {sample0.shape}"
    t_axis, n_layers, n_heads, ctx_len = sample0.shape
    assert n_layers == num_layers and n_heads == num_heads, (
        f"Expected {num_layers} layers and {num_heads} heads, got {n_layers} and {n_heads}"
    )

    attn = sample0.detach().float().cpu().numpy()

    n_rows = math.ceil(math.sqrt(n_heads))
    n_cols = math.ceil(n_heads / n_rows)

    fig_width = n_cols * 3.2
    fig_height = n_rows * 3.2
    fig = plt.figure(figsize=(fig_width, fig_height))
    gs = gridspec.GridSpec(
        n_rows, n_cols, figure=fig, wspace=0.33, hspace=0.29, left=0.04, right=0.97, top=0.92, bottom=0.08
    )

    for i in range(n_heads):
        row, col = divmod(i, n_cols)
        ax = fig.add_subplot(gs[row, col])
        attn_t = attn[:, layer, i, :].T
        im = ax.imshow(attn_t, aspect="equal", origin="lower", extent=(0, t_axis, 0, ctx_len), cmap="bone_r", vmax=None)
        ax.set_xlabel("Prediction Timestep", fontsize=10)
        ax.set_xticks(np.arange(0, t_axis + 128, 128))
        ax.set_ylabel("Context Timestep", fontsize=10)
        ax.set_yticks(np.arange(0, ctx_len, 128))
        ax.set_title(f"Head {i}", fontsize=11)
        ax.tick_params(labelsize=8)
        cbar = fig.colorbar(im, ax=ax, shrink=0.85)
        cbar.ax.tick_params(labelsize=7)
        cbar.set_label("attention", fontsize=8)

    fig.suptitle(f"Attention Heatmaps (Layer {layer})", fontsize=17, fontweight="bold", y=0.98)
    plt.savefig(os.path.join(plot_save_dir, f"attention_heatmap_L{layer}_all_heads.pdf"), bbox_inches="tight", dpi=200)
    plt.show()


plot_attention_heatmap_all_heads(sample0, cfg.plot_save_dir, layer=0, num_layers=num_layers, num_heads=num_heads)


In [ ]:
def collect_head_outputs(pipeline: CircuitLensChronos, num_layers: int, num_heads: int) -> dict:
    return {
        i: [collect_attributions(pipeline.ca_head_attribution_outputs[i][j]) for j in range(num_heads)]
        for i in range(num_layers)
    }


head_outputs = collect_head_outputs(pipeline, num_layers, num_heads)

In [ ]:
pipeline.ca_v_attribution_outputs[0][0][0].shape

In [ ]:
head_outputs[0][0].shape

In [ ]:
def plot_ca_v_attribution_layers(
    pipeline: CircuitLensChronos,
    *,
    layer_start: int,
    layer_end_exclusive: int,
    num_heads: int,
    prediction_length: int,
    plot_save_dir: str,
    rollout: int = 0,
    sample_idx: int = 0,
) -> None:
    os.makedirs(plot_save_dir, exist_ok=True)
    for layer in range(layer_start, layer_end_exclusive):
        fig, axes = plt.subplots(3, 4, figsize=(24, 24))
        axes = axes.flatten()

        for head in range(num_heads):
            ax = axes[head]

            logits = (
                pipeline.unembed_residual(pipeline.ca_v_attribution_outputs[layer][head][rollout][sample_idx])
                .detach()
                .cpu()
                .float()
                .numpy()
            )
            vabs = np.nanmax(np.abs(logits)) if np.any(np.isfinite(logits)) else 0

            im = ax.imshow(
                logits[:prediction_length, :].T,
                aspect="auto",
                cmap="RdBu",
                vmin=-vabs,
                vmax=vabs,
            )

            ax.invert_yaxis()
            ax.set_xlabel("Prediction Timestep")
            ax.set_ylabel("Vocab Index")
            ax.set_title(f"Layer {layer}, Head {head}")

            cbar = fig.colorbar(im, ax=ax)
            cbar.set_label("Logit Value")

        plt.suptitle(f"Cross-Attention Value Attribution (Layer {layer})", fontsize=16, y=1.00)
        plt.tight_layout()
        plt.savefig(os.path.join(plot_save_dir, f"ca_v_attribution_L{layer}_all_heads.png"), dpi=200)
        plt.close(fig)


plot_ca_v_attribution_layers(
    pipeline,
    layer_start=3,
    layer_end_exclusive=4,
    num_heads=num_heads,
    prediction_length=cfg.prediction_length,
    plot_save_dir=cfg.plot_save_dir,
    rollout=0,
    sample_idx=0,
)

In [ ]:
def mean_attention_entropy_per_head(attention_scores: torch.Tensor) -> np.ndarray:
    """Mean over prediction timesteps of per-head attention entropy, shape [layers, heads]."""
    attn = attention_scores.detach().float().cpu().numpy()
    eps = 1e-12
    p = attn / (attn.sum(axis=-1, keepdims=True) + eps)
    ent_t_l_h = -(p * np.log(p + eps)).sum(axis=-1)
    return ent_t_l_h.mean(axis=0)


avg_ent_l_h = mean_attention_entropy_per_head(sample0)

In [ ]:
def plot_top_bottom_entropy_head_heatmaps(
    sample0: torch.Tensor,
    avg_ent_l_h: np.ndarray,
    plot_save_dir: str,
    *,
    k: int = 9,
    imshow_vmax: float = 0.1,
) -> tuple[list[tuple[int, int]], list[tuple[int, int]]]:
    import math

    import matplotlib.gridspec as gridspec

    attn = sample0.detach().float().cpu().numpy()
    t_axis, _n_layers, n_heads, ctx_len = attn.shape

    entropy_flat = avg_ent_l_h.reshape(-1)
    indices_flat = np.argsort(entropy_flat)
    top_k_indices = indices_flat[-k:]
    bottom_k_indices = indices_flat[:k]

    bottom_k_heads = [(idx // n_heads, idx % n_heads) for idx in bottom_k_indices]
    top_k_heads = [(idx // n_heads, idx % n_heads) for idx in top_k_indices[::-1]]

    n_rows = math.ceil(math.sqrt(k))
    n_cols = math.ceil(k / n_rows)

    fig_width = 2 * n_cols * 3.2
    fig_height = n_rows * 3.2
    fig = plt.figure(figsize=(fig_width, fig_height))
    total_cols = n_cols * 2

    gs = gridspec.GridSpec(
        n_rows, total_cols, figure=fig, wspace=0.35, hspace=0.31, left=0.03, right=0.98, top=0.9, bottom=0.07
    )

    x_left = (n_cols / total_cols) / 2.0
    x_right = (n_cols + n_cols / 2.0) / total_cols

    fig.text(x_left, 0.94, "Lowest Entropy Heads", ha="center", fontsize=16, fontweight="bold")
    fig.text(x_right, 0.94, "Highest Entropy Heads", ha="center", fontsize=16, fontweight="bold")

    for i, (layer_i, head_i) in enumerate(bottom_k_heads):
        row, col = divmod(i, n_cols)
        ax = fig.add_subplot(gs[row, col])
        attn_t = attn[:, layer_i, head_i, :].T
        vmax = np.max(attn_t)
        print(f"Bottom heads vmax: {vmax}")
        im = ax.imshow(
            attn_t, aspect="equal", origin="lower", extent=(0, t_axis, 0, ctx_len), cmap="bone_r", vmax=imshow_vmax
        )
        ax.set_xlabel("Prediction Timestep", fontsize=9)
        ax.set_xticks(np.arange(0, t_axis + 128, 128))
        ax.set_ylabel("Context Timestep", fontsize=9)
        ax.set_yticks(np.arange(0, ctx_len, 128))
        ax.set_title(f"Layer {layer_i}, Head {head_i}\n(Entropy: {avg_ent_l_h[layer_i, head_i]:.2f})", fontsize=10)
        ax.tick_params(labelsize=8)
        cbar = fig.colorbar(im, ax=ax, shrink=0.8, extend="max")
        cbar.ax.tick_params(labelsize=7)
        cbar.ax.text(
            1.25,
            1.1,
            f"max = {vmax:.2f}",
            ha="left",
            va="center",
            fontsize=5,
            transform=cbar.ax.transAxes,
            clip_on=False,
        )

    for i, (layer_i, head_i) in enumerate(top_k_heads):
        row, col = divmod(i, n_cols)
        ax = fig.add_subplot(gs[row, col + n_cols])
        attn_t = attn[:, layer_i, head_i, :].T
        vmax = np.max(attn_t)
        print(f"Top heads vmax: {vmax}")
        im = ax.imshow(
            attn_t, aspect="equal", origin="lower", extent=(0, t_axis, 0, ctx_len), cmap="bone_r", vmax=imshow_vmax
        )
        ax.set_xlabel("Prediction Timestep", fontsize=9)
        ax.set_xticks(np.arange(0, t_axis + 128, 128))
        ax.set_ylabel("Context Timestep", fontsize=9)
        ax.set_yticks(np.arange(0, ctx_len, 128))
        ax.set_title(f"Layer {layer_i}, Head {head_i}\n(Entropy: {avg_ent_l_h[layer_i, head_i]:.2f})", fontsize=10)
        ax.tick_params(labelsize=8)
        cbar = fig.colorbar(im, ax=ax, shrink=0.8, extend="max")
        cbar.ax.tick_params(labelsize=7)
        cbar.ax.text(
            1.25,
            1.1,
            f"max = {vmax:.2f}",
            ha="left",
            va="center",
            fontsize=5,
            transform=cbar.ax.transAxes,
            clip_on=False,
        )

    plt.savefig(
        os.path.join(plot_save_dir, f"attention_heatmap_top_bottom_{k}_entropy_heads.pdf"), bbox_inches="tight", dpi=200
    )
    plt.show()

    return bottom_k_heads, top_k_heads


bottom_k_heads, top_k_heads = plot_top_bottom_entropy_head_heatmaps(sample0, avg_ent_l_h, cfg.plot_save_dir, k=9)

print(f"Bottom {len(bottom_k_heads)} entropy heads: {bottom_k_heads}")
print(f"Top {len(top_k_heads)} entropy heads: {top_k_heads}")